In [24]:
import langchain
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.document_loaders import PyPDFLoader
from langchain.retrievers import ContextualCompressionRetriever, SVMRetriever, TFIDFRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from transformers import AutoTokenizer

In [2]:
CHROMA_STORAGE_PATH = "../data/vectors/chroma"

In [3]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:4B",
    temperature=0,
    verbose=True,
    extract_reasoning=True,
)

embedding = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")

vector_db = Chroma(
    embedding_function=embedding,
    persist_directory=CHROMA_STORAGE_PATH,
)
vector_db._collection.count()

/tmp/ipykernel_141726/1026824824.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_db = Chroma(


278

In [4]:
texts = [
    """The Amanita phalloides has a large and imposing epigeous (aboveground) fruiting body (basidiocarp).""",
    """A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.""",
    """A. phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.""",
]
question = "Tell me about all-white mushrooms with large fruiting bodies"

smalldb = Chroma.from_texts(texts, embedding=embedding)

In [5]:
smalldb.similarity_search(question, k=2)

[Document(metadata={}, page_content='A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.'),
 Document(metadata={}, page_content='The Amanita phalloides has a large and imposing epigeous (aboveground) fruiting body (basidiocarp).')]

In [6]:
smalldb.max_marginal_relevance_search(question, k=2, fetch_k=3)

[Document(metadata={}, page_content='A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.'),
 Document(metadata={}, page_content='A. phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.')]

In [7]:
question = "what did they say about matlab?"
docs_ss = vector_db.similarity_search(question, k=3)
docs_ss

[Document(metadata={'page_label': '9', 'page': 8, 'creationdate': '2008-07-11T11:25:23-07:00', 'author': '', 'total_pages': 22, 'creator': 'PScript5.dll Version 5.2.2', 'source': '../data/pdf/cs229_lectures/MachineLearning-Lecture01.pdf', 'moddate': '2008-07-11T11:25:23-07:00', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'title': ''}, page_content='machine learning stuff was actually useful. So what was it that you learned? Was it \nlogistic regression? Was it the PCA? Was it the data networks? What was it that you \nlearned that was so helpful?" And the student said, "Oh, it was the MATLAB."  \nSo for those of you that don\'t know MATLAB yet, I hope you do learn it. It\'s not hard, \nand we\'ll actually have a short MATLAB tutorial in one of the discussion sections for \nthose of you that don\'t know it.  \nOkay. The very last piece of logistical thing is the discussion sections. So discussion \nsections will be taught by the TAs, and attendance at discussion sections is optional

In [8]:
docs_mmrs = vector_db.max_marginal_relevance_search(question, k=3)
docs_mmrs

[Document(metadata={'total_pages': 22, 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creationdate': '2008-07-11T11:25:23-07:00', 'page': 8, 'page_label': '9', 'title': '', 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2008-07-11T11:25:23-07:00', 'source': '../data/pdf/cs229_lectures/MachineLearning-Lecture01.pdf', 'author': ''}, page_content='machine learning stuff was actually useful. So what was it that you learned? Was it \nlogistic regression? Was it the PCA? Was it the data networks? What was it that you \nlearned that was so helpful?" And the student said, "Oh, it was the MATLAB."  \nSo for those of you that don\'t know MATLAB yet, I hope you do learn it. It\'s not hard, \nand we\'ll actually have a short MATLAB tutorial in one of the discussion sections for \nthose of you that don\'t know it.  \nOkay. The very last piece of logistical thing is the discussion sections. So discussion \nsections will be taught by the TAs, and attendance at discussion sections is optional

### Addressing Specificity: working with metadata

In [9]:
question = "what did they say about regression in the third lecture?"
docs = vector_db.similarity_search(
    question, k=3, filter={"source": "../data/pdf/cs229_lectures/MachineLearning-Lecture03.pdf"}
)
docs

[Document(metadata={'title': '', 'source': '../data/pdf/cs229_lectures/MachineLearning-Lecture03.pdf', 'moddate': '2008-07-11T11:25:03-07:00', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'page': 0, 'creator': 'PScript5.dll Version 5.2.2', 'page_label': '1', 'author': '', 'creationdate': '2008-07-11T11:25:03-07:00', 'total_pages': 16}, page_content='MachineLearning-Lecture03  \nInstructor (Andrew Ng):Okay. Good morning and welcome back to the third lecture of \nthis class. So here’s what I want to do today, and some of the topics I do today may seem \na little bit like I’m jumping, sort of, from topic to topic, but here’s, sort of, the outline for \ntoday and the illogical flow of ideas. In the last lecture, we talked about linear regression \nand today I want to talk about sort of an adaptation of that called locally weighted \nregression. It’s very a popular algorithm that’s actually one of my former mentors \nprobably favorite machine learning algorithm.  \nWe’ll then talk about

### Addressing Specificity: working with metadata using self-query retriever

In [10]:
metadata_field_info = [
    AttributeInfo(
        name="source",
        description="The lecture the chunk is from, should be one of `../data/pdf/cs229_lectures/MachineLearning-Lecture01.pdf`, `../data/pdf/cs229_lectures/MachineLearning-Lecture02.pdf`, or `../data/pdf/cs229_lectures/MachineLearning-Lecture03.pdf`",
        type="string",
    ),
    AttributeInfo(
        name="page",
        description="The page from the lecture",
        type="integer",
    ),
]

In [11]:
question = "what did they say about regression in the third lecture?"

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vector_db,
    document_contents="Lecture notes",
    metadata_field_info=metadata_field_info,
    verbose=True,
)

In [12]:
langchain.debug = True
docs = retriever.invoke(question)
langchain.debug = False

docs

[chain/start] [retriever:SelfQueryRetriever > chain:query_constructor] Entering Chain run with input:
{
  "query": "what did they say about regression in the third lecture?"
}
[chain/start] [retriever:SelfQueryRetriever > chain:query_constructor > prompt:FewShotPromptTemplate] Entering Prompt run with input:
{
  "query": "what did they say about regression in the third lecture?"
}
[chain/end] [retriever:SelfQueryRetriever > chain:query_constructor > prompt:FewShotPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [retriever:SelfQueryRetriever > chain:query_constructor > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "Human: Your goal is to structure the user's query to match the request schema provided below.\n\n<< Structured Request Schema >>\nWhen responding use a markdown code snippet with a JSON object formatted in the following schema:\n\n```json\n{\n    \"query\": string \\ text string to compare to document contents\n    \"filter\": strin

[Document(metadata={'creationdate': '2008-07-11T11:25:03-07:00', 'source': '../data/pdf/cs229_lectures/MachineLearning-Lecture03.pdf', 'total_pages': 16, 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'author': '', 'page_label': '5', 'moddate': '2008-07-11T11:25:03-07:00', 'title': '', 'page': 4, 'creator': 'PScript5.dll Version 5.2.2'}, page_content='underfitting. You can still run into the same problems with locally weighted regression. \nWhat you just said about – and so some of these things I’ll leave you to discover for \nyourself in the homework problem. You’ll actually see what you just mentioned. Yeah?  \nStudent:It almost seems like you’re not even thoroughly [inaudible] with this locally \nweighted, you had all the data that you originally had anyway.'),
 Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'total_pages': 16, 'moddate': '2008-07-11T11:25:03-07:00', 'creator': 'PScript5.dll Version 5.2.2', 'page': 2, 'source': '../data/pdf/cs229_lectures/Machi

### Additional tricks: compression

In [21]:
def pretty_print_docs(docs):
    for i, doc in enumerate(docs):
        print("=" * 100)
        print(f"Document {i + 1}")
        print(doc.page_content.replace("Extracted relevant parts:  \n", ""))
    # print(f"\n{'-' * 100}\n".join([f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]))

In [16]:
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vector_db.as_retriever(search_type="mmr"),
)

In [19]:
question = "what did they say about matlab?"
compressed_docs = compression_retriever.invoke(question)

In [22]:
pretty_print_docs(compressed_docs)

Document 1
" And the student said, "Oh, it was the MATLAB."  
So for those of you that don't know MATLAB yet, I hope you do learn it. It's not hard, and we'll actually have a short MATLAB tutorial in one of the discussion sections for those of you that don't know it."
Document 2
"those homeworks will be done in either MATLAB or in Octave, which is sort of — I know some people call it a free version of MATLAB, which it sort of is, sort of isn't. So I guess for those of you that haven't seen MATLAB before, and I know most of you have, MATLAB is I guess part of the programming language that makes it very easy to write codes using matrices, to write code for numerical routines, to move data around, to plot data. And it's sort of an extremely easy to learn tool to use for implementing a lot of learning algorithms. And it has somewhat fewer features than MATLAB, but it's free, and for the purposes of this class, it will work for just about everything."
Document 3
"But I think MATLAB is actua

## Other types of retrieval

In [28]:
loader = PyPDFLoader("../data/pdf/cs229_lectures/MachineLearning-Lecture01.pdf")
pages = loader.load()
# all_page_text = [page.page_content for page in pages]
# joined_page_text = " ".join(all_page_text)

In [29]:
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=256,
    chunk_overlap=32,
)
splits = text_splitter.split_documents(docs)
print(len(splits))
splits[:5]

4


[Document(metadata={'creationdate': '2008-07-11T11:25:03-07:00', 'source': '../data/pdf/cs229_lectures/MachineLearning-Lecture03.pdf', 'total_pages': 16, 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'author': '', 'page_label': '5', 'moddate': '2008-07-11T11:25:03-07:00', 'title': '', 'page': 4, 'creator': 'PScript5.dll Version 5.2.2'}, page_content='underfitting. You can still run into the same problems with locally weighted regression. \nWhat you just said about – and so some of these things I’ll leave you to discover for \nyourself in the homework problem. You’ll actually see what you just mentioned. Yeah?  \nStudent:It almost seems like you’re not even thoroughly [inaudible] with this locally \nweighted, you had all the data that you originally had anyway.'),
 Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'total_pages': 16, 'moddate': '2008-07-11T11:25:03-07:00', 'creator': 'PScript5.dll Version 5.2.2', 'page': 2, 'source': '../data/pdf/cs229_lectures/Machi

In [37]:
question = "what did they say about matlab?"
svm_retriever = SVMRetriever.from_documents(splits, embeddings=embedding)
tfidf_retriever = TFIDFRetriever.from_documents(splits)

In [38]:
results = svm_retriever.get_relevant_documents(question)
print(results[0].page_content)

Student:It’s the lowest it –  
Instructor (Andrew Ng):No, exactly. Right. So zero to the same, this is not the same, 
right? And the reason is, in logistic regression this is different from before, right? The 
definition of this H subscript theta of XI is not the same as the definition I was using in 
the previous lecture. And in particular this is no longer theta transpose XI. This is not a 
linear function anymore. This is a logistic function of theta transpose XI. Okay? So even 
though this looks cosmetically similar, even though this is similar on the surface, to the 
Bastrian descent rule I derived last time for least squares regression this is actually a 
totally different learning algorithm. Okay? And it turns out that there’s actually no 
coincidence that you ended up with the same learning rule. We’ll actually talk a bit more 
about this later when we talk about generalized linear models. But this is one of the most 
elegant generalized learning models that we’ll see later. Th

In [39]:
results = tfidf_retriever.get_relevant_documents(question)
print(results[0].page_content)

regression problem like this. What I want to do today is talk about a class of algorithms 
called non-parametric learning algorithms that will help to alleviate the need somewhat 
for you to choose features very carefully. Okay? And this leads us into our discussion of 
locally weighted regression. And just to define the term, linear regression, as we’ve 
defined it so far, is an example of a parametric learning algorithm. Parametric learning 
algorithm is one that’s defined as an algorithm that has a fixed number of parameters that 
fit to the data. Okay? So in linear regression we have a fix set of parameters theta, right? 
That must fit to the data. In contrast, what I’m gonna talk about now is our first non-
parametric learning algorithm. The formal definition, which is not very intuitive, so I’ve 
replaced it with a second, say, more intuitive. The, sort of, formal definition of the non-
parametric learning algorithm is that it’s an algorithm where the number of parameters 
goes w